In [1]:
import anndata as ad
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from calculate_scnetwork_precision_recall import calculate_scnetwork_precision_recall
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"


In [2]:
DATASET = "retinal"
BASE    = f"../../../data/processed/{DATASET}"

rna      = ad.read_h5ad(f"output/{DATASET}_with_networks_seed42.h5ad")
oob_df = rna.obsm["wScReNI_oob_r2"]


In [3]:
gene_r2 = oob_df.mean(axis=0)
cell_r2 = oob_df.mean(axis=1)
networks = np.load(f"output/{DATASET}_networks_seed42.npy")

In [4]:
scNetworks = {
    "CSN": networks,
    "wScReNI": networks
}

In [5]:
ground_truth = pd.read_csv(
    "../../../data/processed/mmp9.TSV.5kb_TF_target.df.txt",
    sep="\t"
)

ground_truth.head()

,TF,Target_genes
1,Acss2,Cldn34d
2,Acss2,Kpna1
3,Acss2,Il31ra
4,Acss2,Vmn2r102
5,Acss2,Pcdha12


In [6]:
genes_in_network = set(rna.var_names.tolist())

ground_truth_filtered = ground_truth[
    ground_truth["Target_genes"].isin(genes_in_network) &
    ground_truth["TF"].isin(genes_in_network)
]

print(f"Filtered gold standard pairs: {len(ground_truth_filtered)}")
print(f"Unique TFs in network: {ground_truth_filtered['TF'].nunique()}")
print(f"Unique targets in network: {ground_truth_filtered['Target_genes'].nunique()}")

tf_target_reference = (
    ground_truth_filtered["Target_genes"].astype(str)
    + "_"
    + ground_truth_filtered["TF"].astype(str)
).tolist()

Filtered gold standard pairs: 5896
Unique TFs in network: 28
Unique targets in network: 461


In [7]:
genes = rna.var_names.tolist()

In [8]:
print(len(networks))
print(networks[0].shape)
print(len(genes))

400
(500, 500)
500


In [9]:
print(len(genes))
print(networks[0].shape)

500
(500, 500)


In [10]:
networks_df = [
    pd.DataFrame(n, index=genes, columns=genes)
    for n in networks
]

In [11]:
scNetworks = {
    "CSN": networks_df,
    "wScReNI": networks_df
}

In [12]:
results = calculate_scnetwork_precision_recall(
    scNetworks=scNetworks,
    TF_target_pair=tf_target_reference,
    top_number=(0, 1000),
    gene_id_gene_name_pair=None,
    gene_name_type=None
)

In [13]:
results[0].head()

,scNetwork_type,precision,recall
0,CSN,0.023229,0.116859
1,CSN,0.019916,0.077680
2,CSN,0.023092,0.108379
3,CSN,0.021055,0.081411
4,CSN,0.026123,0.171811


In [14]:
for k, df in results.items():
    df.to_csv(f"output/precision_recall_k{k}.csv")

In [15]:
mean_precision_k0 = results[0]["precision"].mean()
print(mean_precision_k0)

0.03181370624551638


In [16]:
print(f"Gold standard pairs: {len(tf_target_reference)}")
print(f"Unique TFs: {ground_truth['TF'].nunique()}")
print(f"Unique targets: {ground_truth['Target_genes'].nunique()}")

Gold standard pairs: 5896
Unique TFs: 656
Unique targets: 22987


In [17]:
df_k1000 = results[0][results[0]["scNetwork_type"] == "wScReNI"].copy()

print("Precision unique values:", df_k1000["precision"].nunique())
print("Recall unique values:", df_k1000["recall"].nunique())
print(df_k1000[["precision", "recall"]].describe())
print(df_k1000.head(10))

Precision unique values: 400
Recall unique values: 336
        precision      recall
count  400.000000  400.000000
mean     0.031814    0.178360
std      0.005798    0.050445
min      0.017932    0.070217
25%      0.028856    0.153621
50%      0.033231    0.184023
75%      0.035664    0.214467
max      0.048416    0.281038
  scNetwork_type  precision    recall
0        wScReNI   0.023229  0.116859
1        wScReNI   0.019916  0.077680
2        wScReNI   0.023092  0.108379
3        wScReNI   0.021055  0.081411
4        wScReNI   0.026123  0.171811
5        wScReNI   0.020687  0.098202
6        wScReNI   0.021823  0.083955
7        wScReNI   0.036323  0.186906
8        wScReNI   0.022956  0.106174
9        wScReNI   0.043625  0.254579


In [18]:
df_k1000 = results[0][results[0]["scNetwork_type"] == "wScReNI"].copy()

df_k1000 = df_k1000.reset_index(drop=True)

df_k1000["rf_r2"] = cell_r2.values

print(df_k1000[["precision", "recall", "rf_r2"]].head(10))
print(f"\nPrecision unique: {df_k1000['precision'].nunique()}")
print(f"RF R² unique: {df_k1000['rf_r2'].nunique()}")

from scipy import stats
for metric in ["precision", "recall"]:
    r, p = stats.spearmanr(df_k1000["rf_r2"], df_k1000[metric])
    print(f"Spearman {metric}: r={r:.4f}, p={p:.2e}")

   precision    recall     rf_r2
0   0.023229  0.116859 -0.098477
1   0.019916  0.077680 -0.152509
2   0.023092  0.108379 -0.112454
3   0.021055  0.081411 -0.133888
4   0.026123  0.171811 -0.109528
5   0.020687  0.098202 -0.104038
6   0.021823  0.083955 -0.146750
7   0.036323  0.186906 -0.125989
8   0.022956  0.106174 -0.099534
9   0.043625  0.254579 -0.106592

Precision unique: 400
RF R² unique: 400
Spearman precision: r=0.0257, p=6.08e-01
Spearman recall: r=0.3909, p=4.70e-16


In [19]:
print(results[0]["recall"].mean())

0.1783599050203528
